# GenCare AI finetuning
---
Auteur:   Eva Rombouts  
Datum:    26-12-2023.  
Doel:     Ervaring opdoen met Python, NLP  
Doel project: Voorspellen onrustscores met een llm  

Voor dit script wordt gebruik gemaakt van data gegenereerd met de OpenAI API.  

---

## Setup

In [1]:
import pandas as pd
import numpy as np
from transformers import BertTokenizer, BertModel, pipeline
import torch.nn as nn
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
import torch
from tqdm import tqdm
from torch.optim import AdamW
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [2]:
# Parameters
seed = 6
max_len = 512  # Max lengte van een tekst voor BERT
batch_size = 16
learning_rate = 2e-5
epochs = 3

In [3]:
gci = pd.read_csv('../zorgdata/gci_rapportages.csv')

In [4]:
gci.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1402 entries, 0 to 1401
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliënt_id    1402 non-null   object 
 1   week         1402 non-null   int64  
 2   dag          1402 non-null   object 
 3   niveau       1402 non-null   object 
 4   rapportage   1402 non-null   object 
 5   onrustscore  1395 non-null   float64
dtypes: float64(1), int64(1), object(4)
memory usage: 65.8+ KB


In [5]:
# Er zijn 7 rijen zonder onrustscore. Verwijder:
gci.dropna(inplace=True)
gci.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1395 entries, 0 to 1401
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   cliënt_id    1395 non-null   object 
 1   week         1395 non-null   int64  
 2   dag          1395 non-null   object 
 3   niveau       1395 non-null   object 
 4   rapportage   1395 non-null   object 
 5   onrustscore  1395 non-null   float64
dtypes: float64(1), int64(1), object(4)
memory usage: 76.3+ KB


In [6]:
# Eerste split: 80% train, 20% tijdelijk (val+test)
df_train, df_temp = train_test_split(gci, test_size=0.2, random_state=seed)
# Tweede split: split de 20% tijdelijk in 10% val, 10% test
df_val, df_test = train_test_split(df_temp, test_size=0.5, random_state=seed)

print(f'Trainset: {len(df_train)}')
print(f'Validatieset: {len(df_val)}')
print(f'Testset: {len(df_test)}')

Trainset: 1116
Validatieset: 139
Testset: 140


De taak die we willen gaan ondernemen: Onrustscores voorspellen op basis van de teksten. 
Dit is in essentie een vorm van regressie, met het doel om een continue waarde te voorspellen. Hiervoor moet een transformermodel eerst worden aangepast voor regressie. 

Modelkeuze: Mogelijkheden zijn BERT, RoBERTa, of DistilBERT. Deze modellen zijn al getraind op grote tekstcorpora en hebben een goed begrip van taalstructuren.
Om te beginnen start ik met BERT: 

In [7]:
model_naam = 'wietsedv/bert-base-dutch-cased'
tokenizer = BertTokenizer.from_pretrained(model_naam)
model = BertModel.from_pretrained(model_naam)

Some weights of BertModel were not initialized from the model checkpoint at wietsedv/bert-base-dutch-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Om een idee te krijgen van de structuur van het model is het van belang om de architectuur te bestuderen. Dat gebeurt hieronder. 
Todo: Neem de tijd om het te bestuderen. 

In [8]:
model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30073, 768, padding_idx=3)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

### Beschrijving BERT architectuur
(Uitleg aangepast van ChatGPT)

Deze architectuur is van het model BertForSequenceClassification, een variant van BERT (Bidirectional Encoder Representations from Transformers) die specifiek is ontworpen voor sequentieclassificatietaken. Hier zijn de belangrijkste eigenschappen:

**BERT Model**: Het is een transformer-model dat gebruikmaakt van de transformer-architectuur, vooral bekend om zijn vermogen om contextuele relaties tussen woorden in een tekst te begrijpen. BERT is pre-trained op een grote hoeveelheid tekst en kan worden aangepast aan specifieke taken (zoals classificatie).

**Embeddings**:
- Woordembeddings: Deze vertegenwoordigen individuele woorden in een dichte vectorruimte. De dimensie van deze embeddings is 768.
- Positie-embeddings: BERT neemt volgorde-informatie op door gebruik te maken van positie-embeddings. Deze zijn ook van dimensie 768.
- Token Type Embeddings: Worden gebruikt voor het onderscheiden van verschillende zinnen of taken in een input.
- Layer Normalization en Dropout: Deze worden gebruikt voor normalisatie en om overfitting te voorkomen.

**BertEncoder**: Het hart van het model, bestaande uit 12 transformer-lagen (voor de basisvariant van BERT). Elke laag heeft:
- BertAttention: Een mechanisme dat de modelinvoer 'bekijkt' en bepaalt welke delen van de invoer belangrijk zijn.
- BertIntermediate: Een feedforward netwerk dat de dimensie tijdelijk verhoogt van 768 naar 3072 voor meer complexe representaties.
- BertOutput: Converteert de dimensie terug van 3072 naar 768 en past nog een laagnormalisatie en dropout toe.

**BertPooler**: Een laag die de output van de laatste encoder-laag transformeert en zorgt voor een vaste outputgrootte, wat belangrijk is voor classificatiedoeleinden.

Deze architectuur is krachtig voor sequentieclassificatie vanwege zijn vermogen om rijke, contextuele representaties van tekst te genereren, waardoor het model subtiele nuances en complexe patronen in tekstgegevens kan begrijpen en classificeren.

### Aanpassen voor Regressie
Hoewel deze modellen doorgaans worden gebruikt voor classificatie of andere NLP-taken, kunnen ze worden aangepast voor regressie. Dit houdt meestal in dat de laatste laag (een classificatielaag) wordt vervangen door een regressielaag (een lineaire laag die één output produceert).

In [9]:
class BertForRegression(nn.Module):
    def __init__(self, bert_model):
        super(BertForRegression, self).__init__()
        self.bert = bert_model
        # De regressielaag: BERT's outputgrootte (768 voor base-modellen) naar 1
        self.regression = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):
        # De output van BERT
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Neem de output van de laatste verborgen laag voor de [CLS] token
        cls_output = outputs.last_hidden_state[:, 0, :]
        # Pas de regressielaag toe
        return self.regression(cls_output)

# Maak een instantie van het aangepaste model
regression_model = BertForRegression(model)


In [10]:
class RapportageDataset(Dataset):
    def __init__(self, teksten, scores, tokenizer, max_len):
        self.teksten = teksten
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.teksten)

    def __getitem__(self, item):
        tekst = str(self.teksten[item])
        score = self.scores[item]

        encoding = self.tokenizer.encode_plus(
            tekst,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'text': tekst,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'score': torch.tensor(score, dtype=torch.float)
        }

In [11]:
# Datasets voorbereiden
train_data = RapportageDataset(
    teksten=df_train['rapportage'].to_numpy(),
    scores=df_train['onrustscore'].to_numpy(),
    tokenizer=tokenizer,
    max_len=max_len
)

val_data = RapportageDataset(
    teksten=df_val['rapportage'].to_numpy(),
    scores=df_val['onrustscore'].to_numpy(),
    tokenizer=tokenizer,
    max_len=max_len
)

test_data = RapportageDataset(
    teksten=df_test['rapportage'].to_numpy(),
    scores=df_test['onrustscore'].to_numpy(),
    tokenizer=tokenizer,
    max_len=max_len
)

# DataLoaders
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_data, batch_size=batch_size)
test_loader = DataLoader(test_data, batch_size=batch_size)

In [12]:
# Setup van het model en optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
regression_model.to(device)
optimizer = AdamW(regression_model.parameters(), lr=learning_rate)

# Loss functie
loss_fn = nn.MSELoss()

In [14]:
# Trainingsloop
for epoch in range(epochs):
    regression_model.train()
    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        scores = batch['score'].to(device)

        outputs = regression_model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, scores.unsqueeze(1))
        
        loss.backward()
        optimizer.step()

        loop.set_description(f'Epoch {epoch}')
        loop.set_postfix(loss=loss.item())


Epoch 2: 100%|██████████| 70/70 [18:18<00:00, 15.70s/it, loss=1.02e+3]


In [15]:
# Opslaan van de modelparameters (aanbevolen methode)
torch.save(regression_model.state_dict(), '../zorgdata/bert_regression_model.pth')


Training: U moet het model trainen met uw specifieke dataset. Dit betekent het voeden van de teksten aan het model en het laten leren van de overeenkomstige onrustscores. Dit proces omvat het definiëren van een geschikte loss-functie voor regressie, zoals Mean Squared Error (MSE) of Mean Absolute Error (MAE).

Gegevensvoorbereiding: Zorg ervoor dat uw gegevens goed zijn voorbereid en geformatteerd. Dit betekent het tokeniseren van de teksten op een manier die compatibel is met het gekozen model en het normaliseren van uw onrustscores (indien nodig).

Evaluatie: Beoordeel de prestaties van uw model met behulp van geschikte regressiemetrieken, zoals MSE, MAE, of R-squared.

Fijnafstelling en Optimalisatie: Mogelijk moet u het model finetunen en de hyperparameters optimaliseren om de beste prestaties voor uw specifieke toepassing te bereiken.

Gebruik van een Domeinspecifiek Model: Als er modellen beschikbaar zijn die specifiek zijn getraind op medische of zorggerelateerde gegevens, kunnen deze modellen beter presteren vanwege hun domeinspecifieke kennis.

Een concrete implementatie zou kunnen beginnen met een model als bert-base-uncased (of een Nederlandstalige variant indien uw teksten in het Nederlands zijn), waarbij u vervolgens een regressielaag toevoegt en het model traint met uw dataset van zorgrapportages en bijbehorende onrustscores.

Let's go...  
Stap 1  

Voor het voorspellen van een onrustscore, die in essentie een vorm van regressie is (waarbij het doel is om een continue waarde te voorspellen), zou u een getransformeerd taalmodel kunnen gebruiken dat is aangepast voor regressietaken. Hier zijn een paar stappen en overwegingen:

Modelkeuze: Begin met een voorgeleerd taalmodel zoals BERT, RoBERTa, of DistilBERT. Deze modellen zijn al getraind op grote tekstcorpora en hebben een goed begrip van taalstructuren.


In [16]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [17]:
def evaluate_model(model, data_loader, device):
    model.eval()
    voorspellingen = []
    werkelijke_scores = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            scores = batch['score'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            voorspellingen.extend(outputs.flatten().cpu().numpy())
            werkelijke_scores.extend(scores.flatten().cpu().numpy())

    mse = mean_squared_error(werkelijke_scores, voorspellingen)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(werkelijke_scores, voorspellingen)

    return mse, rmse, mae


In [18]:
mse, rmse, mae = evaluate_model(regression_model, test_loader, device)

In [19]:
print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'MAE: {mae}')


MSE: 1125.177978515625
RMSE: 33.54367446899414
MAE: 27.785545349121094


MSE: 1206.82568359375
RMSE: 34.73939514160156
MAE: 28.972118377685547